In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator

In [ ]:
os.makedirs("outputs/figures", exist_ok=True)

In [12]:
#load the raw data

df_raw = pd.read_csv(r"../data/raw/ACLED_Data_2026-04-17.csv")

In [13]:
## Configure pandas display options for better readability
# ── Config ────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

## Data Exploration

In [15]:
# ── 1. DATA STRUCTURE OVERVIEW ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. MISSING VALUES")
print("=" * 60)

missing = (
    df_raw.isnull()
    .sum()
    .rename("n_missing")
    .to_frame()
    .assign(pct_missing=lambda x: (x["n_missing"] / len(df_raw) * 100).round(2))
    .sort_values("n_missing", ascending=False)
)
missing_nonzero = missing[missing["n_missing"] > 0]

print(f"\nColumns with no missing    : {(missing['n_missing'] == 0).sum()}")
print(f"Columns with missing values: {len(missing_nonzero)}")
print(f"\nDetail (only columns with missing values):\n")
print(missing_nonzero.to_string())


2. MISSING VALUES

Columns with no missing    : 22
Columns with missing values: 13

Detail (only columns with missing values):

                    n_missing  pct_missing
admin3                  41670       100.00
tags                    33530        80.47
assoc_actor_2           32032        76.87
assoc_actor_1           30852        74.04
civilian_targeting      24904        59.76
inter2                   8915        21.39
actor2                   8915        21.39
population_best          8900        21.36
population_1km           7096        17.03
population_2km           7066        16.96
population_5km           7060        16.94
admin1                      1         0.00
admin2                      1         0.00


In [16]:
cols_with_missing = missing_nonzero.index.tolist()

if not cols_with_missing:
    print("\nNo missing values found — correlation analysis skipped.")
else:
    print("\n" + "=" * 60)
    print("3. MISSING CORRELATION WITH YEAR AND REGION")
    print("=" * 60)

    # -- 3a. By year
    print("\n--- 3a. Missing rate by year (columns with > 0 missing) ---\n")
    missing_by_year = (
        df_raw.assign(**{
            f"_miss_{c}": df_raw[c].isnull().astype(int)
            for c in cols_with_missing
        })
        .groupby("year")[[f"_miss_{c}" for c in cols_with_missing]]
        .mean()
        .mul(100)
        .round(2)
    )
    missing_by_year.columns = cols_with_missing
    print(missing_by_year.to_string())

    # -- 3b. By region (admin1)
    if "admin1" in df_raw.columns:
        print("\n--- 3b. Missing rate by region (admin1) ---\n")
        missing_by_region = (
            df_raw.assign(**{
                f"_miss_{c}": df_raw[c].isnull().astype(int)
                for c in cols_with_missing
            })
            .groupby("admin1")[[f"_miss_{c}" for c in cols_with_missing]]
            .mean()
            .mul(100)
            .round(2)
        )
        missing_by_region.columns = cols_with_missing
        mask = (missing_by_region > 1).any(axis=1)
        print(missing_by_region[mask].sort_values(
            cols_with_missing[0], ascending=False
        ).to_string())


3. MISSING CORRELATION WITH YEAR AND REGION

--- 3a. Missing rate by year (columns with > 0 missing) ---

      admin3    tags  assoc_actor_2  assoc_actor_1  civilian_targeting  inter2  actor2  population_best  population_1km  population_2km  population_5km  admin1  admin2
year                                                                                                                                                                   
1997   100.0  100.00          88.03          86.62               70.42   14.08   14.08           100.00          100.00          100.00          100.00    0.00    0.00
1998   100.0   99.34          87.50          84.87               67.76   25.00   25.00           100.00          100.00          100.00          100.00    0.00    0.00
1999   100.0   97.04          88.18          77.34               71.43   18.72   18.72           100.00          100.00          100.00          100.00    0.00    0.00
2000   100.0   98.80          82.53          87.95   

In [ ]:
# ── 4. VISUALIZATIONS ─────────────────────────────────────────────────────────

# ── Fig 1: missing values per column ─────────────────────────────────────────
if not cols_with_missing:
    print("\nNo missing values to plot.")
else:
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing_nonzero) * 0.45)))
    sns.barplot(
        data=missing_nonzero.reset_index(),
        x="pct_missing", y="index",
        color="#4878d0", ax=ax
    )
    ax.set_xlabel("% missing")
    ax.set_ylabel("")
    ax.set_title("% of missing values per column")
    ax.xaxis.set_major_locator(MaxNLocator(5))
    for bar, val in zip(ax.patches, missing_nonzero["pct_missing"]):
        ax.text(
            bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9
        )
    plt.tight_layout()
    plt.savefig("outputs/figures/01_dataexploration_fig1_missing_per_column.png", bbox_inches="tight")
    plt.show()

# ── Fig 2: event count per year ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
events_per_year = df_raw.groupby("year").size().reset_index(name="n_events")
sns.barplot(data=events_per_year, x="year", y="n_events", color="#4878d0", ax=ax)
ax.set_xlabel("Year")
ax.set_ylabel("Number of events")
ax.set_title("Event count per year (Nigeria, ACLED)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig2_events_per_year.png", bbox_inches="tight")
plt.show()

# ── Fig 3: missing rate over time (one subplot per column with missing) ───────
if cols_with_missing:
    n_cols = len(cols_with_missing)
    fig, axes = plt.subplots(n_cols, 1, figsize=(12, 3.5 * n_cols), squeeze=False)
    for i, col in enumerate(cols_with_missing):
        rate = (
            df_raw.assign(_miss=df_raw[col].isnull().astype(int))
            .groupby("year")["_miss"]
            .mean()
            .mul(100)
            .reset_index(name="pct_missing")
        )
        sns.lineplot(data=rate, x="year", y="pct_missing",
                     marker="o", color="#e07b54", ax=axes[i][0])
        axes[i][0].set_title(f"Missing rate over time — '{col}'")
        axes[i][0].set_xlabel("Year")
        axes[i][0].set_ylabel("% missing")
        axes[i][0].axhline(rate["pct_missing"].mean(), ls="--",
                           color="gray", linewidth=0.8, label="overall mean")
        axes[i][0].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("outputs/figures/01_dataexploration_fig3_missing_over_time.png", bbox_inches="tight")
    plt.show()

# ── Fig 4: missing rate by state (admin1) ────────────────────────────────────
if cols_with_missing and "admin1" in df_raw.columns:
    col_focus = cols_with_missing[0]
    miss_reg = (
        df_raw.assign(_miss=df_raw[col_focus].isnull().astype(int))
        .groupby("admin1")["_miss"]
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .reset_index(name="pct_missing")
    )
    fig, ax = plt.subplots(figsize=(10, max(4, len(miss_reg) * 0.35)))
    sns.barplot(data=miss_reg, x="pct_missing", y="admin1",
                color="#4878d0", ax=ax)
    ax.set_xlabel("% missing")
    ax.set_ylabel("State (admin1)")
    ax.set_title(f"Missing rate of '{col_focus}' by state")
    plt.tight_layout()
    plt.savefig("outputs/figures/01_dataexploration_fig4_missing_by_region.png", bbox_inches="tight")
    plt.show()

# ── Fig 5: heatmap — missing rate by year × state ────────────────────────────
if cols_with_missing and "admin1" in df_raw.columns:
    col_focus = cols_with_missing[0]
    pivot = (
        df_raw.assign(_miss=df_raw[col_focus].isnull().astype(int))
        .groupby(["year", "admin1"])["_miss"]
        .mean()
        .mul(100)
        .unstack("admin1")
        .fillna(0)
    )
    fig, ax = plt.subplots(figsize=(18, max(5, len(pivot) * 0.4)))
    sns.heatmap(
        pivot, ax=ax, cmap="YlOrRd",
        linewidths=0.3, linecolor="white",
        cbar_kws={"label": "% missing"}
    )
    ax.set_title(f"Missing rate of '{col_focus}' — year × state")
    ax.set_xlabel("State (admin1)")
    ax.set_ylabel("Year")
    plt.tight_layout()
    plt.savefig("outputs/figures/01_dataexploration_fig5_heatmap_year_state.png", bbox_inches="tight")
    plt.show()

print("\nAnalysis complete. Figures saved to current directory.")

In [ ]:
df_raw

From this, so far the only conclusion would be dropping admin3, since it's the only one with a high missing value


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

PALETTE = "muted"
TOP_N = 15  # for high-cardinality fields like actor1


# ── 1. OVERALL DISTRIBUTIONS ──────────────────────────────────────────────────
print("=" * 60)
print("1. CATEGORICAL VARIABLE DISTRIBUTIONS")
print("=" * 60)

cat_cols = ["event_type", "sub_event_type", "actor1", "assoc_actor_1", "actor2", "interaction"]

for col in cat_cols:
    if col not in df_raw.columns:
        print(f"\n[SKIP] Column '{col}' not found.")
        continue
    vc = df_raw[col].value_counts(dropna=False)
    print(f"\n--- {col} ({df_raw[col].nunique(dropna=True)} unique values) ---")
    print(vc.head(20).to_string())

# ── 2. DISTRIBUTION PLOTS — OVERALL ──────────────────────────────────────────
print("\n" + "=" * 60)
print("2. DISTRIBUTION PLOTS")
print("=" * 60)

# -- event_type
fig, ax = plt.subplots(figsize=(10, 4))
vc = df_raw["event_type"].value_counts()
sns.barplot(x=vc.values, y=vc.index, color="#4878d0", ax=ax)
ax.set_title("Event type distribution")
ax.set_xlabel("Count")
ax.set_ylabel("")
for bar, val in zip(ax.patches, vc.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig6_event_type.png", bbox_inches="tight")
plt.show()

# -- sub_event_type
fig, ax = plt.subplots(figsize=(10, 6))
vc = df_raw["sub_event_type"].value_counts()
sns.barplot(x=vc.values, y=vc.index, color="#4878d0", ax=ax)
ax.set_title("Sub-event type distribution")
ax.set_xlabel("Count")
ax.set_ylabel("")
for bar, val in zip(ax.patches, vc.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig7_sub_event_type.png", bbox_inches="tight")
plt.show()

# -- interaction
fig, ax = plt.subplots(figsize=(10, 5))
vc = df_raw["interaction"].value_counts()
sns.barplot(x=vc.values, y=vc.index.astype(str), color="#4878d0", ax=ax)
ax.set_title("Interaction code distribution")
ax.set_xlabel("Count")
ax.set_ylabel("Interaction code")
for bar, val in zip(ax.patches, vc.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig8_interaction.png", bbox_inches="tight")
plt.show()

# -- actor1 (top N)
fig, ax = plt.subplots(figsize=(10, 6))
vc = df_raw["actor1"].value_counts().head(TOP_N)
sns.barplot(x=vc.values, y=vc.index, color="#e07b54", ax=ax)
ax.set_title(f"Top {TOP_N} actor1")
ax.set_xlabel("Count")
ax.set_ylabel("")
for bar, val in zip(ax.patches, vc.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig9_actor1.png", bbox_inches="tight")
plt.show()

# -- assoc_actor_1 (top N, excluding NaN)
fig, ax = plt.subplots(figsize=(10, 6))
vc = df_raw["assoc_actor_1"].value_counts(dropna=True).head(TOP_N)
sns.barplot(x=vc.values, y=vc.index, color="#e07b54", ax=ax)
ax.set_title(f"Top {TOP_N} assoc_actor_1 (non-null)")
ax.set_xlabel("Count")
ax.set_ylabel("")
for bar, val in zip(ax.patches, vc.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig10_assoc_actor1.png", bbox_inches="tight")
plt.show()

# -- actor2 (top N, excluding NaN)
fig, ax = plt.subplots(figsize=(10, 6))
vc = df_raw["actor2"].value_counts(dropna=True).head(TOP_N)
sns.barplot(x=vc.values, y=vc.index, color="#e07b54", ax=ax)
ax.set_title(f"Top {TOP_N} actor2 (non-null)")
ax.set_xlabel("Count")
ax.set_ylabel("")
for bar, val in zip(ax.patches, vc.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig11_actor2.png", bbox_inches="tight")
plt.show()

# ── 3. DISTRIBUTION BY LOCATION (admin1) ─────────────────────────────────────
print("\n" + "=" * 60)
print("3. DISTRIBUTION BY STATE (admin1)")
print("=" * 60)

# -- event count per state
fig, ax = plt.subplots(figsize=(10, 8))
vc = df_raw["admin1"].value_counts()
sns.barplot(x=vc.values, y=vc.index, color="#4878d0", ax=ax)
ax.set_title("Event count by state (admin1)")
ax.set_xlabel("Count")
ax.set_ylabel("")
for bar, val in zip(ax.patches, vc.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig12_events_by_state.png", bbox_inches="tight")
plt.show()

# -- event_type × admin1 heatmap (normalized by row to show composition)
pivot_et = (
    df_raw.groupby(["admin1", "event_type"])
    .size()
    .unstack("event_type")
    .fillna(0)
)
pivot_et_pct = pivot_et.div(pivot_et.sum(axis=1), axis=0).mul(100).round(1)
pivot_et_pct = pivot_et_pct.loc[df_raw["admin1"].value_counts().index]  # sort by total events

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot_et_pct, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "% of state events"}, ax=ax)
ax.set_title("Event type composition by state (% of state total)")
ax.set_xlabel("Event type")
ax.set_ylabel("State (admin1)")
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig13_event_type_by_state_heatmap.png", bbox_inches="tight")
plt.show()

# -- sub_event_type × admin1 heatmap (top 10 states by event count)
top10_states = df_raw["admin1"].value_counts().head(10).index
pivot_sub = (
    df_raw[df_raw["admin1"].isin(top10_states)]
    .groupby(["admin1", "sub_event_type"])
    .size()
    .unstack("sub_event_type")
    .fillna(0)
)
pivot_sub_pct = pivot_sub.div(pivot_sub.sum(axis=1), axis=0).mul(100).round(1)

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(pivot_sub_pct, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "% of state events"}, ax=ax)
ax.set_title("Sub-event type composition — top 10 states (% of state total)")
ax.set_xlabel("Sub-event type")
ax.set_ylabel("State (admin1)")
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig14_sub_event_type_top10_states.png", bbox_inches="tight")
plt.show()

# -- top actor1 per state (top 5 states, top 5 actors each)
top5_states = df_raw["admin1"].value_counts().head(5).index
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=False)
for ax, state in zip(axes, top5_states):
    vc = (
        df_raw[df_raw["admin1"] == state]["actor1"]
        .value_counts()
        .head(8)
    )
    sns.barplot(x=vc.values, y=vc.index, color="#e07b54", ax=ax)
    ax.set_title(state, fontsize=10)
    ax.set_xlabel("Count")
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=8)
plt.suptitle("Top 8 actor1 — 5 most affected states", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("outputs/figures/01_dataexploration_fig15_actor1_top5_states.png", bbox_inches="tight")
plt.show()

print("\nExploration complete. Figures saved to current directory.")

## Export Processed Data

In [ ]:
# Drop admin3 (100% missing — no usable data)
df_processed = df_raw.drop(columns=["admin3"])

os.makedirs("../data/processed", exist_ok=True)
df_processed.to_csv("../data/processed/nigeria_acled.csv", index=False)
print(f"Saved {len(df_processed):,} rows × {df_processed.shape[1]} columns → data/processed/nigeria_acled.csv")